In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
path = "/home/jovyan/mri-data/test-results"

filenames_csv = [
    "Revision MRI I - Part I_results.csv",
    "Revision MRI I - Part II_results.csv",
    "Test Revision MRI I - Part III - A_results.csv",
    "Test Revision MRI I - Part III - B_results.csv",
    "Test Revision MRI I - Part III - C_results.csv",
    "Lecture 01_results.csv",
    "Lecture 02_results.csv",
    "Lecture 03_results.csv",
]


filenames_png = [
    "Revision MRI I - Part I_results.png",
    "Revision MRI I - Part II_results.png",
    "Test Revision MRI I - Part III - A_results.png",
    "Test Revision MRI I - Part III - B_results.png",
    "Test Revision MRI I - Part III - C_results.png",
    "Lecture 01_results.png",
    "Lecture 02_results.png",
    "Lecture 03_results.png",
]

total_max_points_per_test = [
    120,
    120,
    40,
    40,
    40,
    20,
    20,
    20,
]

In [ ]:
for filename_csv, filename_png, total_max_points in zip(filenames_csv, filenames_png, total_max_points_per_test):
    
    # Load the CSV file
    data = pd.read_csv(
        os.path.join(path, filename_csv),
        delimiter=";",
        skipinitialspace=True,
        on_bad_lines="warn",
    )
    
    # Clean the data: drop rows with all NaN values
    data_clean = data.dropna(how="all")
    
    # Convert 'Durchlauf' to numeric, errors='coerce' will turn non-numeric into NaN
    data_clean["Durchlauf"] = pd.to_numeric(data_clean["Durchlauf"], errors="coerce")
    
    # For each 'Zähler', keep only the row with the highest 'Durchlauf'
    latest_trials = data_clean.loc[data_clean.groupby("Zähler")["Durchlauf"].idxmax()]
    
    # Extract total points and points for each question
    # Total points column: 'Testergebnis in Punkten'
    # Question columns: 'Vorlesung 01 - Frage 01' to 'Vorlesung 01 - Frage 10'
    
    percentile_top_50_percent = (
        np.sum(latest_trials["Testergebnis in Punkten"].dropna() > 0.5 * total_max_points)
        / latest_trials["Testergebnis in Punkten"].dropna().size
        * 100
    )
    
    percentile_top_66_percent = (
        np.sum(latest_trials["Testergebnis in Punkten"].dropna() > 0.66 * total_max_points)
        / latest_trials["Testergebnis in Punkten"].dropna().size
        * 100
    )
    
    total_bins = np.linspace(-0.5, total_max_points + 0.5, num=total_max_points + 1)
    question_bins = np.linspace(-4.5, total_max_points + 0.5, num=total_max_points + 1)
    
    # Plot histogram for total points
    plt.figure(figsize=(10, 6))
    plt.hist(
        latest_trials["Testergebnis in Punkten"].dropna(),
        bins=total_bins,
        edgecolor="black",
    )
    plt.title("Histogram of Total Points (Latest Trial per Person)")
    plt.xlabel("Total Points")
    plt.ylabel("Frequency")
    plt.ylim([0, 10])
    plt.grid(True)
    
    plt.plot(
        [total_max_points * 0.5, total_max_points * 0.5],
        [0, 10],
        "--",
        label=f"{np.round(percentile_top_50_percent)}% of participants achieved more than 50% of the maximum points"
    )
    
    plt.plot(
        [total_max_points * 0.66, total_max_points * 0.66],
        [0, 10],
        "--",
        label=f"{np.round(percentile_top_66_percent)}% of participants achieved more than 66% of the maximum points"
    )
    
    plt.legend(loc="upper center", fontsize=12)
    
    plt.savefig(
        os.path.join(path, filename_png),
        format="png",
        dpi=200,
        bbox_inches="tight",
    )
    # plt.close()
    
    # # Plot histograms for each question
    # question_columns = [
    #     col for col in latest_trials.columns if col.startswith("Vorlesung 01 - Frage")
    # ]
    #
    # for question in question_columns:
    #     plt.figure(figsize=(10, 6))
    #     plt.hist(latest_trials[question].dropna(), bins=question_bins, edgecolor='black')
    #     plt.title(f'Histogram of Points for {question} (Latest Trial per Person)')
    #     plt.xlabel('Points')
    #     plt.ylabel('Frequency')
    #     plt.grid(True)
    #     # plt.savefig(f'{question}_histogram.png', format='png', dpi=200, bbox_inches='tight')
    #     # plt.close()